In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle, RegularPolygon, Polygon
from matplotlib.backends.backend_pdf import PdfPages
import random
import os
import torch
import dill
from torchmetrics.classification import BinaryAccuracy

from sparse_generalization.scripts.shape_dataset import is_same_row, generate_grid_row, generate_grid_adjacent, generate_grid_row_col, generate_grid_same_row
from sparse_generalization.data.shapes.constants import SHAPE_COLORS, SHAPES_TO_IDX, SHAPES, IDX_TO_SHAPES, SHAPE_MAP
from sparse_generalization.utils.util_funcs import positionalencoding2d
from sparse_generalization.models.spartan import SPARTAN

%load_ext autoreload
%autoreload 2


In [4]:
# Shapes and mapping
shapes = ["circle","square","triangle","diamond","pentagon","heart","star","moon","plus"]
shape_to_idx = {s:i for i,s in enumerate(shapes)}
shape_colors = {
    "circle": "red",
    "square": "blue",
    "triangle": "green",
    "diamond": "purple",
    "pentagon": "orange",
    "heart": "pink",
    "star": "gold",
    "moon": "black",
    "plus": "cyan"
}

def draw_shape(ax, shape, x, y, size=0.35):
    color = shape_colors[shape]
    if shape == "circle":
        ax.add_patch(Circle((x, y), size, color=color))
    elif shape == "square":
        ax.add_patch(Rectangle((x-size, y-size), 2*size, 2*size, color=color))
    elif shape == "triangle":
        ax.add_patch(RegularPolygon((x,y), 3, radius=size, color=color))
    elif shape == "diamond":
        verts = [(x,y+size),(x+size,y),(x,y-size),(x-size,y)]
        ax.add_patch(Polygon(verts, color=color))
    elif shape == "pentagon":
        ax.add_patch(RegularPolygon((x,y),5,radius=size,color=color))
    elif shape == "heart":
        t = np.linspace(0,2*np.pi,100)
        hx = size*16*np.sin(t)**3
        hy = size*(13*np.cos(t)-5*np.cos(2*t)-2*np.cos(3*t)-np.cos(4*t))
        hx = hx/18 + x
        hy = hy/18 + y
        ax.add_patch(Polygon(np.column_stack([hx,hy]), color=color))
    elif shape == "star":
        pts=[]
        for i in range(10):
            angle=i*np.pi/5
            r=size if i%2==0 else size*0.4
            pts.append((x+r*np.cos(angle), y+r*np.sin(angle)))
        ax.add_patch(Polygon(pts,color=color))
    elif shape == "moon":
        moon = Circle((x,y), size, color=color)
        ax.add_patch(moon)
        cover = Circle((x+size*0.4,y+size*0.1), size, color="white")
        ax.add_patch(cover)
    elif shape == "plus":
        t = size*0.6
        ax.add_patch(Rectangle((x-t/2,y-size), t,2*size, color=color))
        ax.add_patch(Rectangle((x-size,y-t/2), 2*size,t, color=color))

# Generate random grid
def generate_random_grid():
    rows, cols = 3,3
    grid = np.empty((rows,cols),dtype=object)
    remaining = shapes.copy()
    random.shuffle(remaining)
    idx = 0
    for r in range(rows):
        for c in range(cols):
            grid[r,c] = remaining[idx]
            idx += 1
    return grid

# Horizontal adjacency check
def is_horizontal_adjacent(grid, shape1, shape2):
    for r in range(3):
        for c in range(2):
            if grid[r,c] == shape1 and grid[r,c+1] == shape2:
                return True
            if grid[r,c] == shape2 and grid[r,c+1] == shape1:
                return True
    return False


In [ ]:
# Dataset generation, equal amount of each label.
num_images = 250
half_images = num_images // 2
save_dir = "../data/shapes/"
mode = 'train' #other options: test_a, test_b
func = 'row_col'
size = 5
os.makedirs(save_dir, exist_ok=True)

all_tensors = []
labels = []
label_counts = {"a":0, "b":0}

i = 0
attempts = 0
max_attempts = num_images * 10

pdf_path = os.path.join(save_dir, f"all_grids_{mode}_{func}.pdf")
pdf_pages = PdfPages(pdf_path)

for label in [True, False]:
    for _ in range(half_images):
        grid = generate_grid_row_col(size=size, label_A=label, mode=mode)

        # Create figure and save to PDF
        fig, ax = plt.subplots(figsize=(5,5))
        for r in range(size):
            for c in range(size):
                x = c + 0.5
                y = size - r - 0.5
                color = SHAPE_COLORS[grid[r,c]]
                obj = SHAPE_MAP[grid[r,c]](x, y, color=color)
                obj.draw(ax)
                
        ax.text(1.5, 0.0, f"Label: {label}", fontsize=16, ha='center', va='bottom', color="black", weight='bold')
        ax.set_xlim(0,size)
        ax.set_ylim(0,size)
        ax.set_aspect("equal")
        ax.axis("off")
        
        pdf_pages.savefig(fig)
        plt.close(fig)
        i += 1

pdf_pages.close()

# Save tensors and labels
# np.save(os.path.join(save_dir,f"grid_tensors_{data_type}.npy"), np.array(all_tensors))
# np.save(os.path.join(save_dir,f"grid_labels_{data_type}.npy"), np.array(labels))

In [11]:
save_dir = "../data/shapes/"
with open(save_dir + 'shapes_train_size5.pl', 'rb') as file:
    dataset = dill.load(file)
    file.close()

midpoint = dataset["X_train"].size(0) // 2
X_train_pos = dataset["X_train"][:midpoint]
Y_train_pos = dataset["Y_train"][:midpoint]
X_train_neg = dataset["X_train"][midpoint:]
Y_train_neg = dataset["Y_train"][midpoint:]

In [13]:
# Dataset generation, equal amount of each label.

size = 5
i = 0

pdf_path = os.path.join(save_dir, f"train_grids_neg.pdf")
pdf_pages = PdfPages(pdf_path)

for sample, label in zip(X_train_neg[:250], Y_train_neg[:250]):
    print(i, end='\r')
    # Create figure and save to PDF
    fig, ax = plt.subplots(figsize=(5,5))
    for r in range(size):
        for c in range(size):
            x = c + 0.5
            y = size - r - 0.5
            shape = IDX_TO_SHAPES[sample[r,c].item()]
            color = SHAPE_COLORS[shape]
            obj = SHAPE_MAP[shape](x, y, color=color)
            obj.draw(ax)
            
    ax.text(1.5, 0.0, f"Label: {label.item()}", fontsize=16, ha='center', va='bottom', color="black", weight='bold')
    ax.set_xlim(0,size)
    ax.set_ylim(0,size)
    ax.set_aspect("equal")
    ax.axis("off")
    
    pdf_pages.savefig(fig)
    plt.close(fig)
    i += 1

pdf_pages.close()


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sparse_generalization.models.spartan import SPARTAN

# ── dummy data ────────────────────────────────────────────────────────────────
B, W, H = 16, 4, 4          # batch, grid width, height
x_dummy = torch.randint(0, 64, (B, W, H, 1)).float()
y_dummy = torch.randint(0, 2,  (B, 1)).float()
loader  = DataLoader(TensorDataset(x_dummy, y_dummy), batch_size=B)

# ── build model ───────────────────────────────────────────────────────────────
model = SPARTAN(
    agg_pool      = False,
    include_sparsity = True,
    path_sparsity = True,
    model_dim=32, 
    alpha         = 0.1,
    num_layers    = 1,
    layernorm=True,
    residual      = True,
    embedding_inp = True,
    device        = "cpu",
).to("cpu")

# ── hook: record grad norms per module ───────────────────────────────────────
grad_log = {}   # name -> list of output-grad norms seen during backward

def make_hook(name):
    def hook(module, grad_input, grad_output):
        norms = []
        for g in grad_output:
            if g is not None:
                norms.append(g.detach().abs().mean().item())
        if norms:
            grad_log.setdefault(name, []).extend(norms)
    return hook

handles = []
for name, module in model.named_modules():
    if name:   # skip the root module itself
        h = module.register_full_backward_hook(make_hook(name))
        handles.append(h)

# ── also hook each Parameter directly for fine-grained view ──────────────────
param_grad_log = {}

def param_hook(name):
    def hook(grad):
        param_grad_log[name] = grad.detach().abs().mean().item()
    return hook

param_handles = []
for name, param in model.named_parameters():
    if param.requires_grad:
        h = param.register_hook(param_hook(name))
        param_handles.append(h)

# ── one forward + backward pass ───────────────────────────────────────────────
model.train()
x, y = next(iter(loader))
out, masks, attns = model(x)

rec_loss    = model.loss(out, y)
sparse_loss = model._enforce_sparsity(masks)
loss        = rec_loss + sparse_loss

model.optimizer.zero_grad()
loss.backward()

# ── print results ─────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  rec_loss={rec_loss.item():.4f}  sparse_loss={sparse_loss.item():.4f}")
print(f"{'='*60}\n")

# -- module-level output grads (coarse) --
print(f"{'MODULE':<45}  {'MEAN |GRAD|':>12}")
print("-" * 60)
for name, norms in sorted(grad_log.items(), key=lambda kv: -max(kv[1])):
    print(f"  {name:<43}  {max(norms):>12.2e}")

# -- per-parameter grads (fine) --
print(f"\n{'PARAMETER':<50}  {'MEAN |GRAD|':>12}")
print("-" * 65)
for name, norm in sorted(param_grad_log.items(), key=lambda kv: -kv[1]):
    marker = " ◀ DEAD" if norm < 1e-8 else ""
    print(f"  {name:<48}  {norm:>12.2e}{marker}")

# ── clean up hooks ────────────────────────────────────────────────────────────
for h in handles + param_handles:
    h.remove()


  rec_loss=0.7032  sparse_loss=0.4141

MODULE                                          MEAN |GRAD|
------------------------------------------------------------
  loss                                             1.00e+00
  out                                              3.16e-02
  layers.0.mha                                     3.14e-04
  layers.0                                         3.14e-04
  layers.0.mlp.3                                   1.89e-04
  layers.0.mlp                                     1.89e-04
  layers.0.mlp.2                                   1.50e-04
  feature_map.5                                    1.33e-04
  feature_map                                      1.33e-04
  layers.0.mha.projection                          1.17e-04
  feature_map.4                                    7.04e-05
  layers.0.mlp.0                                   6.69e-05
  layers.0.mlp.1                                   6.02e-05
  feature_map.3                                    5.41e-05

/var/folders/h5/1rtlmq9x41bgl8btn0x1p6g80000gn/T/ipykernel_62106/3489096337.py:69: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()
